# Stage 10A: Linear Regression Baseline

**Project context:** SPY next-day high-volatility risk alert. This homework predicts continuous `next_day_abs_return` as a transparent diagnostic baseline. It does not replace the final time-aware classification model.

**Timing rule:** each predictor ends at date `t`; the target is the next trading day's absolute return. The split is chronological: the newest 20% is held out as future-like test data.


## 1. Reproducible setup and Stage09 feature snapshot


In [1]:
from pathlib import Path
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

ROOT = Path.cwd()
if not (ROOT / "homework" / "homework10a").is_dir():
    for candidate in (ROOT, *ROOT.parents):
        if (candidate / "homework" / "homework10a").is_dir():
            ROOT = candidate
            break
HOMEWORK = ROOT / "homework" / "homework10a"
RAW = HOMEWORK / "data" / "raw"
PROCESSED = HOMEWORK / "data" / "processed"
REPORTS = HOMEWORK / "reports"
for directory in [PROCESSED, REPORTS]:
    directory.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)
print("Homework directory:", HOMEWORK)


Repository root: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng
Homework directory: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/homework/homework10a


## 2. Time-safe target, features, and chronological split


In [2]:
snapshot_path = RAW / "spy_feature_candidates_stage09_snapshot.csv"
features_raw = pd.read_csv(snapshot_path, parse_dates=["date"])

baseline_features = [
    "abs_return_t",
    "intraday_range_t",
    "log_volume_change_t",
    "rolling_volatility_5_t",
]
target = "next_day_abs_return"
model_frame = features_raw.loc[:, ["date", *baseline_features, "stress_interaction_t", target]].copy()
model_frame["intraday_range_sq"] = model_frame["intraday_range_t"] ** 2
model_frame = model_frame.dropna(subset=[*baseline_features, "stress_interaction_t", target]).reset_index(drop=True)

split_index = int(len(model_frame) * 0.80)
train = model_frame.iloc[:split_index].copy()
test = model_frame.iloc[split_index:].copy()

assert features_raw.shape == (2512, 18)
assert model_frame["date"].is_monotonic_increasing
assert train["date"].max() < test["date"].min()
assert len(train) + len(test) == len(model_frame)

print("Model-ready observations:", len(model_frame))
print("Train range:", train["date"].min().date(), "to", train["date"].max().date(), "(", len(train), ")")
print("Test range:", test["date"].min().date(), "to", test["date"].max().date(), "(", len(test), ")")


Model-ready observations: 2506
Train range: 2016-08-30 to 2024-08-16 ( 2004 )
Test range: 2024-08-19 to 2026-08-20 ( 502 )


## 3. Baseline and transformed-feature comparison

The baseline tests a simple linear relationship. The diagnostic variant adds `intraday_range_sq` plus the Stage09 stress interaction. Squared input features are still linear regression because the coefficients remain linear.


In [3]:
specifications = {
    "baseline": baseline_features,
    "range_squared_and_stress": [*baseline_features, "intraday_range_sq", "stress_interaction_t"],
}
results = {}
metric_records = []

for name, columns in specifications.items():
    model = LinearRegression().fit(train[columns], train[target])
    prediction = model.predict(test[columns])
    residual = test[target].to_numpy() - prediction
    results[name] = {"model": model, "prediction": prediction, "residual": residual, "columns": columns}
    metric_records.append({
        "model": name,
        "test_observations": len(test),
        "r2": r2_score(test[target], prediction),
        "rmse": mean_squared_error(test[target], prediction) ** 0.5,
    })

metrics = pd.DataFrame(metric_records).sort_values("rmse").reset_index(drop=True)
metrics_path = PROCESSED / "linear_regression_metrics.csv"
metrics.to_csv(metrics_path, index=False)
display(metrics)


,model,test_observations,r2,rmse
0,baseline,502,0.160573,0.007275
1,range_squared_and_stress,502,0.111528,0.007485


## 4. Residual diagnostics for the selected baseline


In [4]:
selected_name = "baseline"
selected = results[selected_name]
fitted = selected["prediction"]
residuals = selected["residual"]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0, 0].scatter(fitted, residuals, alpha=0.65, color="#2563EB", edgecolor="none")
axes[0, 0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0, 0].set(title="Residuals vs fitted", xlabel="Fitted next-day absolute return", ylabel="Residual")

axes[0, 1].hist(residuals, bins=28, color="#0F766E", edgecolor="white")
axes[0, 1].set(title="Residual histogram", xlabel="Residual", ylabel="Count")

stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("Normal QQ plot")

axes[1, 1].scatter(residuals[:-1], residuals[1:], alpha=0.65, color="#7C3AED", edgecolor="none")
axes[1, 1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1, 1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1, 1].set(title="Lag-1 residual check", xlabel="Residual at prior test day", ylabel="Residual at current test day")

fig.suptitle("Stage10A SPY linear-regression diagnostics", y=1.02)
fig.tight_layout()
diagnostics_path = REPORTS / "linear_regression_diagnostics.png"
fig.savefig(diagnostics_path, dpi=150, bbox_inches="tight")
plt.show()


/var/folders/hy/9nh5rd9526vd43l1zms4t1lh0000gn/T/ipykernel_76284/3513548673.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Interpretation and trust decision

- **Linearity:** inspect residuals-versus-fitted for curvature or systematic structure. The baseline is intentionally simple; the squared-range variant is a targeted diagnostic, not automatic complexity.
- **Independence:** the lag-1 plot is an informal check only. Financial observations can retain regime dependence, which motivates the time-aware classification work in Stage10B.
- **Homoscedasticity and normality:** absolute-return targets are nonnegative and heavy-tailed, so constant variance and Gaussian residuals are unlikely to hold perfectly. These plots make that model risk visible.
- **Metrics:** compare held-out $R^2$ and RMSE only alongside the diagnostics. If the transformed variant is worse, that is evidence not to add it merely because it is more complex.
- **Conclusion:** this is useful as an interpretable continuous-risk baseline, but it is **not trusted as a deployed alert rule**. The final project must use chronological classification, train-only target construction, probability calibration/thresholding, and risk-aware evaluation.


In [5]:
assert metrics.shape == (2, 4)
assert np.isfinite(metrics[["r2", "rmse"]].to_numpy()).all()
assert (metrics["rmse"] > 0).all()
assert diagnostics_path.is_file() and diagnostics_path.stat().st_size > 0
assert metrics_path.is_file() and metrics_path.stat().st_size > 0
print("Stage10A checks passed.")
print("Metrics:", metrics_path.name)
print("Diagnostics:", diagnostics_path.name)


Stage10A checks passed.
Metrics: linear_regression_metrics.csv
Diagnostics: linear_regression_diagnostics.png
